# ESE 5971 — Milestone 2
## Early Warning and Decision Support for Animal Shelter Overcrowding

**Case study:** Austin Animal Center, historical intake/outcome event records.

**Research objective:** Investigate the data-generating process, measurement quality, seasonality, temporal dependence, and a preliminary prediction workflow for shelter population pressure.

**Important scope limitation:** The observed target here is **future seven-day net animal event flow**, not validated onsite occupancy or overcrowding. Onsite counts, offsite animals, and historical capacity require additional evidence before making actual overcrowding claims. This notebook is exploratory and does not estimate intervention effects.

**Decision timing:** At the end of day *t*, after the day's recorded events are available, forecast days *t+1* through *t+7*.

## 1. Environment and configuration
Run cells in order. Both source CSV files are required; no source data are modified.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.metrics import mean_absolute_error, mean_squared_error

H = 7
# Supports running from the repository root or the notebooks/ folder.
search_paths = [Path('../data/raw'), Path('data/raw')]
DATA_DIR = next((p for p in search_paths if (p / 'intakes.csv').is_file() and (p / 'outcomes.csv').is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Place intakes.csv and outcomes.csv in data/raw/; run from repository root or notebooks/.')
pd.set_option('display.max_columns', None)
print('Data folder:', DATA_DIR.resolve())

## 2. Data sources and event-level observations
The official Austin Animal Center historical datasets contain **intake events** and **outcome events**. An Animal ID may appear repeatedly, and one event row is not the same thing as one unique animal or one shelter-day.

- [Austin Animal Center Intakes (historical)](https://data.austintexas.gov/d/wter-evkm)
- [Austin Animal Center Outcomes (historical)](https://data.austintexas.gov/d/9t4d-g238)

In [ ]:
intakes = pd.read_csv(DATA_DIR / 'intakes.csv', low_memory=False)
outcomes = pd.read_csv(DATA_DIR / 'outcomes.csv', low_memory=False)
print('Intakes:', intakes.shape, '| Outcomes:', outcomes.shape)
print('Intake columns:', intakes.columns.tolist())
print('Outcome columns:', outcomes.columns.tolist())
display(intakes.head(5), outcomes.head(5))

## 3. Initial data quality and measurement
Check missing values, duplicate **rows**, recurring Animal IDs, and mixed date formats. Multiple events for one Animal ID are not automatically duplicates; the source timestamps remain unchanged.

In [ ]:
for label, df in [('Intakes', intakes), ('Outcomes', outcomes)]:
    print(f'\n=== {label} ===')
    print('Rows:', len(df), '| Unique animals:', df['Animal ID'].nunique())
    print('Missing Animal IDs:', df['Animal ID'].isna().sum())
    print('Exact duplicate rows:', df.duplicated().sum())
    print('IDs with multiple records:', (df['Animal ID'].value_counts() > 1).sum())
    missing = pd.DataFrame({'missing_n': df.isna().sum(), 'missing_pct': 100*df.isna().mean()})
    display(missing.round(2))

In [ ]:
# The source contains timestamps with and without explicit UTC offsets.
s = outcomes['DateTime'].astype('string')
has_offset = s.str.contains(r'(?:Z|[+-]\d{2}:\d{2})$', regex=True, na=False)
format_summary = pd.DataFrame({'format': ['With UTC offset', 'Without UTC offset'],
                               'count': [int(has_offset.sum()), int((~has_offset).sum())]})
format_summary['percent'] = (format_summary['count'] / len(outcomes)*100).round(2)
display(format_summary)
print('Missing raw outcome timestamps:', outcomes['DateTime'].isna().sum())

**Timestamp caution.** For preliminary daily aggregation, the date as *written* is used. Naive and offset-aware strings must not be silently interpreted as one UTC time zone; the original time-zone convention is not fully verified. Observations near midnight may be assigned differently under a verified local-time conversion.

In [ ]:
# Step 5 — Intake and Outcome Event Types

print("=== INTAKE TYPES ===")
print(
    intakes["Intake Type"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n=== OUTCOME TYPES ===")
print(
    outcomes["Outcome Type"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n=== ANIMAL TYPES ===")
print("Intakes:")
print(
    intakes["Animal Type"]
    .value_counts(dropna=False)
    .to_string()
)

print("\nOutcomes:")
print(
    outcomes["Animal Type"]
    .value_counts(dropna=False)
    .to_string()
)

## 4. Repeated animals and tentative stay matching
ID overlap and event-count consistency are diagnostics only. Sorting each animal’s intakes and outcomes and pairing the *j*-th events creates **candidate pairs**, not verified shelter stays. Negative durations or interleaved events may reflect recording conventions, missing historical events, or invalid pairing. Do not use these pairs to claim onsite occupancy.

In [ ]:
# Step 6 — Intake-Outcome Matching Feasibility

intake_ids = set(intakes["Animal ID"].dropna())
outcome_ids = set(outcomes["Animal ID"].dropna())

matched_ids = intake_ids & outcome_ids
intake_only = intake_ids - outcome_ids
outcome_only = outcome_ids - intake_ids

print("=== ANIMAL ID OVERLAP ===")

print("Unique Intake IDs:", len(intake_ids))
print("Unique Outcome IDs:", len(outcome_ids))
print("IDs in both datasets:", len(matched_ids))
print("Intake-only IDs:", len(intake_only))
print("Outcome-only IDs:", len(outcome_only))

print("\n=== MATCH RATE ===")
print(
    "Intake ID match rate:",
    round(len(matched_ids) / len(intake_ids) * 100, 2), "%"
)

print(
    "Outcome ID match rate:",
    round(len(matched_ids) / len(outcome_ids) * 100, 2), "%"
)

print("\n=== MULTIPLE EVENTS ===")

for name, df in [("Intakes", intakes), ("Outcomes", outcomes)]:
    counts = df["Animal ID"].value_counts()

    print(f"\n{name}")
    print("Maximum events for one animal:", counts.max())
    print("Animals with exactly 1 event:", (counts == 1).sum())
    print("Animals with 2 or more events:", (counts >= 2).sum())

In [ ]:
# Step 7 — Compare event counts by Animal ID

intake_counts = intakes["Animal ID"].value_counts()
outcome_counts = outcomes["Animal ID"].value_counts()

event_counts = pd.DataFrame({
    "n_intakes": intake_counts,
    "n_outcomes": outcome_counts
}).fillna(0).astype(int)

event_counts["difference"] = (
    event_counts["n_intakes"] -
    event_counts["n_outcomes"]
)

print("=== EVENT COUNT COMPARISON ===")

print("Total unique animals:", len(event_counts))

print(
    "Equal counts:",
    (event_counts["difference"] == 0).sum()
)

print(
    "More intakes than outcomes:",
    (event_counts["difference"] > 0).sum()
)

print(
    "More outcomes than intakes:",
    (event_counts["difference"] < 0).sum()
)

print("\n=== DIFFERENCE DISTRIBUTION ===")
print(
    event_counts["difference"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n=== EXAMPLE: MULTIPLE EVENTS ===")

example_ids = event_counts[
    (event_counts["n_intakes"] >= 3) |
    (event_counts["n_outcomes"] >= 3)
].head(10)

display(example_ids)

In [ ]:
# Step 8 — Preliminary chronological matching


# Preserve original records
in_events = intakes.copy()
out_events = outcomes.copy()

# Parse intake timestamps
in_events["event_time"] = pd.to_datetime(
    in_events["DateTime"],
    errors="coerce"
)

# Extract wall-clock timestamps from outcome records.
# This is a provisional representation, not a verified timezone conversion.
out_events["event_time"] = pd.to_datetime(
    out_events["DateTime"].astype("string").str[:19],
    format="%Y-%m-%dT%H:%M:%S",
    errors="coerce"
)

# Sort records within each animal
in_events = in_events.sort_values(
    ["Animal ID", "event_time"]
).copy()

out_events = out_events.sort_values(
    ["Animal ID", "event_time"]
).copy()

# Assign chronological event numbers
in_events["event_number"] = (
    in_events.groupby("Animal ID").cumcount() + 1
)

out_events["event_number"] = (
    out_events.groupby("Animal ID").cumcount() + 1
)

# Diagnostic pairing, not yet confirmed valid stays
pairs = in_events[
    ["Animal ID", "event_number", "event_time", "Animal Type"]
].rename(columns={
    "event_time": "intake_time",
    "Animal Type": "animal_type"
}).merge(
    out_events[
        ["Animal ID", "event_number", "event_time", "Outcome Type"]
    ].rename(columns={"event_time": "outcome_time"}),
    on=["Animal ID", "event_number"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

# Calculate duration for candidate pairs
pairs["length_of_stay_days"] = (
    pairs["outcome_time"] - pairs["intake_time"]
).dt.total_seconds() / 86400

print("=== PAIRING RESULTS ===")
print(pairs["_merge"].value_counts().to_string())

print("\n=== TIME ORDER CHECK ===")
print(
    "Negative durations:",
    (pairs["length_of_stay_days"] < 0).sum()
)

print(
    "Zero durations:",
    (pairs["length_of_stay_days"] == 0).sum()
)

print(
    "Positive durations:",
    (pairs["length_of_stay_days"] > 0).sum()
)

print("\n=== DURATION SUMMARY ===")
print(pairs["length_of_stay_days"].describe())

print("\n=== NEGATIVE DURATION EXAMPLES ===")
display(
    pairs.loc[
        pairs["length_of_stay_days"] < 0,
        ["Animal ID", "event_number", "intake_time",
         "outcome_time", "length_of_stay_days"]
    ].head(10)
)

In [ ]:
# Step 9 — Diagnose negative candidate stay durations

negative = pairs.loc[
    pairs["length_of_stay_days"] < 0
].copy()

negative["same_calendar_day"] = (
    negative["intake_time"].dt.normalize()
    == negative["outcome_time"].dt.normalize()
)

negative["outcome_midnight"] = (
    (negative["outcome_time"].dt.hour == 0) &
    (negative["outcome_time"].dt.minute == 0) &
    (negative["outcome_time"].dt.second == 0)
)

print("=== NEGATIVE DURATION DIAGNOSIS ===")

print("Total negative pairs:", len(negative))

print(
    "Same calendar day:",
    negative["same_calendar_day"].sum()
)

print(
    "Different calendar days:",
    (~negative["same_calendar_day"]).sum()
)

print(
    "Same day AND outcome at midnight:",
    (
        negative["same_calendar_day"] &
        negative["outcome_midnight"]
    ).sum()
)

print("\n=== CROSS-DATE NEGATIVE EXAMPLES ===")

display(
    negative.loc[
        ~negative["same_calendar_day"],
        [
            "Animal ID",
            "event_number",
            "intake_time",
            "outcome_time",
            "length_of_stay_days"
        ]
    ].head(10)
)

## 5. External consistency check: September 2023
Compare recorded event counts with **previously transcribed figures** from an Austin monthly report, including a sensitivity check on the calendar window. Official figures below are contextual reference values and have not been independently extracted by this notebook. Inventory snapshots are not equivalent to monthly event totals; reported onsite counts must not be conflated with total in-care population. Source: [Austin Animal Services reports](https://www.austintexas.gov/animal-services/reports).

In [ ]:
# Step 10 — Compare CSV net flow with official inventory


# Use the calendar date written in the original records.
# This avoids making unverified timezone assumptions.
in_dates = pd.to_datetime(
    intakes["DateTime"],
    errors="coerce"
).dt.normalize()

out_dates = pd.to_datetime(
    outcomes["DateTime"].astype("string").str[:10],
    format="%Y-%m-%d",
    errors="coerce"
)

# Two possible time windows, because inventory
# snapshot times are not yet confirmed.
windows = {
    "Sep 1–30 inclusive": ("2023-09-01", "2023-09-30"),
    "Sep 2–30 inclusive": ("2023-09-02", "2023-09-30")
}

results = []

for window_name, (start, end) in windows.items():
    for animal_type in ["Dog", "Cat", "All"]:
        intake_mask = in_dates.between(start, end)
        outcome_mask = out_dates.between(start, end)

        if animal_type != "All":
            intake_mask &= intakes["Animal Type"].eq(animal_type)
            outcome_mask &= outcomes["Animal Type"].eq(animal_type)

        n_in = int(intake_mask.sum())
        n_out = int(outcome_mask.sum())

        results.append({
            "Window": window_name,
            "Animal Type": animal_type,
            "Intakes": n_in,
            "Outcomes": n_out,
            "Net Flow": n_in - n_out
        })

flow_results = pd.DataFrame(results)

print("=== SEPTEMBER 2023 CSV NET FLOW ===")
display(flow_results)

print("\n=== OFFICIAL INVENTORY CHANGES ===")
print("Onsite population change: +104")
print("Total in-care population change: +47")

In [ ]:
# Step 11 — Compare official monthly counts with CSV

official = pd.DataFrame({
    "Animal Type": ["Dog", "Cat"],
    "Official Intakes": [353, 556],
    "Official Outcomes": [301, 540]
})

csv_month = flow_results[
    flow_results["Window"] == "Sep 1–30 inclusive"
]

csv_month = csv_month[
    csv_month["Animal Type"].isin(["Dog", "Cat"])
]

comparison = official.merge(
    csv_month[["Animal Type", "Intakes", "Outcomes"]],
    on="Animal Type"
)

comparison["Intake Difference"] = (
    comparison["Intakes"] - comparison["Official Intakes"]
)

comparison["Outcome Difference"] = (
    comparison["Outcomes"] - comparison["Official Outcomes"]
)

display(comparison)

## 6. Event-to-day aggregation
Primary analytic scope is dogs and cats. Each row of `daily` represents one calendar day for this single shelter, with *recorded* intake and outcome event counts. All calendar dates are explicitly included; zero-filled gaps mean no events were recorded in these CSVs, not confirmed zero operations. Define `net_flow = intakes - outcomes` (events per day), which is **not** onsite occupancy.

In [ ]:
# Step 12 — Build Daily Shelter Flow Dataset


# 1. Create copies to preserve raw data
in_daily = intakes.copy()
out_daily = outcomes.copy()

# 2. Extract calendar dates
in_daily["date"] = pd.to_datetime(
    in_daily["DateTime"],
    errors="coerce"
).dt.normalize()

# Outcome timestamps have mixed timezone formats.
# For this preliminary analysis, use the date as recorded.
out_daily["date"] = pd.to_datetime(
    out_daily["DateTime"].astype("string").str[:10],
    format="%Y-%m-%d",
    errors="coerce"
)

# 3. Restrict to dogs and cats for primary analysis
in_daily = in_daily[
    in_daily["Animal Type"].isin(["Dog", "Cat"])
]

out_daily = out_daily[
    out_daily["Animal Type"].isin(["Dog", "Cat"])
]

# 4. Count daily intake events by animal type
daily_intakes = (
    in_daily
    .groupby(["date", "Animal Type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Dog", "Cat"], fill_value=0)
    .rename(columns={
        "Dog": "dog_intakes",
        "Cat": "cat_intakes"
    })
)

# 5. Count daily outcome events by animal type
daily_outcomes = (
    out_daily
    .groupby(["date", "Animal Type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Dog", "Cat"], fill_value=0)
    .rename(columns={
        "Dog": "dog_outcomes",
        "Cat": "cat_outcomes"
    })
)

# 6. Combine and preserve every calendar day
date_index = pd.date_range(
    min(daily_intakes.index.min(), daily_outcomes.index.min()),
    max(daily_intakes.index.max(), daily_outcomes.index.max()),
    freq="D"
)

daily = (
    daily_intakes
    .join(daily_outcomes, how="outer")
    .reindex(date_index)
    .fillna(0)
    .astype(int)
)

daily.index.name = "date"

# 7. Create derived variables
daily["total_intakes"] = (
    daily["dog_intakes"] + daily["cat_intakes"]
)

daily["total_outcomes"] = (
    daily["dog_outcomes"] + daily["cat_outcomes"]
)

daily["dog_net_flow"] = (
    daily["dog_intakes"] - daily["dog_outcomes"]
)

daily["cat_net_flow"] = (
    daily["cat_intakes"] - daily["cat_outcomes"]
)

daily["net_flow"] = (
    daily["total_intakes"] - daily["total_outcomes"]
)

print("=== DAILY DATASET ===")
print("Shape:", daily.shape)
print("Start:", daily.index.min())
print("End:", daily.index.max())

display(daily.head(10))

In [ ]:
# Step 12.2 — Aggregation integrity check

print("=== RECORD COUNT VERIFICATION ===")

print("Original dog/cat intake records:", len(in_daily))
print("Aggregated intake records:", daily["total_intakes"].sum())

print("\nOriginal dog/cat outcome records:", len(out_daily))
print("Aggregated outcome records:", daily["total_outcomes"].sum())

print("\n=== DAILY SUMMARY ===")
display(
    daily[
        ["total_intakes", "total_outcomes", "net_flow"]
    ].describe()
)

print("\n=== ZERO-EVENT DAYS ===")
print(
    "Days with no recorded intake or outcome:",
    (
        (daily["total_intakes"] == 0) &
        (daily["total_outcomes"] == 0)
    ).sum()
)

## 7. Descriptive EDA: long-run trends and seasonality
**Questions:** How do intake/outcome volumes evolve, and do recurring monthly peaks appear? The 30-day moving average is descriptive. The 2014–2019 seasonal comparison deliberately excludes partial endpoints and the visibly different post-2020 regime; it does not establish that subsequent years follow the same seasonality.

In [ ]:
# Step 12.3 — Daily Intake and Outcome Trends


# Use 30-day rolling averages for easier visualization
rolling = daily[
    ["total_intakes", "total_outcomes"]
].rolling(window=30, min_periods=30).mean()

plt.figure(figsize=(13, 5))

plt.plot(
    rolling.index,
    rolling["total_intakes"],
    label="Daily Intakes (30-day average)"
)

plt.plot(
    rolling.index,
    rolling["total_outcomes"],
    label="Daily Outcomes (30-day average)"
)

plt.title("Austin Animal Center: Daily Intake and Outcome Trends")
plt.xlabel("Date")
plt.ylabel("Animal Events per Day (30-day rolling average)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Step 13 — Monthly Seasonality Analysis


# Focus on complete pre-pandemic calendar years,
# excluding partial 2013 and the 2020 structural-change period.
pre_2020 = daily.loc["2014-01-01":"2019-12-31"].copy()

pre_2020["month"] = pre_2020.index.month

monthly_seasonality = (
    pre_2020
    .groupby("month")[
        ["total_intakes", "total_outcomes"]
    ]
    .mean()
)

print("=== MONTHLY SEASONALITY (2014–2019) ===")
display(monthly_seasonality.round(2))

plt.figure(figsize=(10, 5))

plt.plot(
    monthly_seasonality.index,
    monthly_seasonality["total_intakes"],
    marker="o",
    label="Average Daily Intakes"
)

plt.plot(
    monthly_seasonality.index,
    monthly_seasonality["total_outcomes"],
    marker="o",
    label="Average Daily Outcomes"
)

plt.xticks(
    range(1, 13),
    ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
)

plt.xlabel("Month")
plt.ylabel("Average Number of Events per Day")
plt.title(
    "Monthly Seasonality of Shelter Intake and Outcomes (2014–2019)"
)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Temporal dependence
**Question:** Are successive daily event counts independent? Plot an ACF for 2014–2019, then subtract the sample month-of-year means to probe whether apparent dependence is partly seasonal. The month-adjusted series is **EDA only**, not a leakage-safe modeling transformation. Training-only estimates would be required for deployment or validation preprocessing.

In [ ]:
# Step 14 — Temporal Dependence Analysis


# Focus on pre-2020 period first
series = daily.loc[
    "2014-01-01":"2019-12-31", "total_intakes"
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw time series
axes[0].plot(series.index, series)
axes[0].set_title("Daily Animal Intakes (2014–2019)")
axes[0].set_xlabel("Date")
axes[0].set_ylabel("Intake Events per Day")

# Autocorrelation
plot_acf(
    series,
    lags=30,
    ax=axes[1],
    zero=False
)

axes[1].set_title("Autocorrelation of Daily Intakes")
axes[1].set_xlabel("Lag (Days)")

plt.tight_layout()
plt.show()

print("Lag-1 autocorrelation:", round(series.autocorr(lag=1), 3))
print("Lag-7 autocorrelation:", round(series.autocorr(lag=7), 3))
print("Lag-14 autocorrelation:", round(series.autocorr(lag=14), 3))

In [ ]:
# Step 15 — Seasonally Adjusted Autocorrelation


series = daily.loc[
    "2014-01-01":"2019-12-31", "total_intakes"
].copy()

# Calculate mean intake for each month
monthly_mean = series.groupby(series.index.month).transform("mean")

# Remove average monthly seasonal pattern
adjusted_series = series - monthly_mean

print("=== AUTOCORRELATION COMPARISON ===")

for lag in [1, 7, 14]:
    print(
        f"Lag {lag}:",
        "Raw =", round(series.autocorr(lag), 3),
        "| Adjusted =", round(adjusted_series.autocorr(lag), 3)
    )

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_acf(
    series,
    lags=30,
    ax=axes[0],
    zero=False
)

axes[0].set_title("Raw Daily Intake ACF")

plot_acf(
    adjusted_series,
    lags=30,
    ax=axes[1],
    zero=False
)

axes[1].set_title("Month-Adjusted Daily Intake ACF")

plt.tight_layout()
plt.show()

## 9. Preliminary future seven-day prediction target
At the end of day *t*, define the **auxiliary** target

\[Y_t^{(7)}=\sum_{h=1}^{7} F_{t+h},\quad F_t=I_t-E_t.\]

Units: net recorded events per seven-day interval. Constructing a historical label from future realized data is valid, but that information may never be passed to the model as a prediction-time feature. Adjacent targets share six of seven days and are dependent even if raw flows were independent. Large positive cumulative flow does not necessarily mean true overcrowding, and net flow can miss interim daily peaks.

In [ ]:
# Step 17.1 — Construct the future 7-day net-flow target


# Preserve the original daily dataset
target_df = daily.copy()

# Number of days in the prediction horizon
H = 7

# Y_t = F_(t+1) + ... + F_(t+7)
target_df["target_7d_net_flow"] = (
    pd.concat(
        [
            target_df["net_flow"].shift(-h)
            for h in range(1, H + 1)
        ],
        axis=1
    )
    .sum(axis=1, min_count=H)
)

print("=== TARGET CONSTRUCTION ===")
print("Original daily observations:", len(target_df))
print("Valid target observations:",
      target_df["target_7d_net_flow"].notna().sum())
print("Unavailable target observations:",
      target_df["target_7d_net_flow"].isna().sum())

display(
    target_df[
        ["net_flow", "target_7d_net_flow"]
    ].head(12)
)

In [ ]:
# Step 17.2 — Verify the target calculation

test_date = pd.Timestamp("2023-09-10")

future_flows = daily.loc[
    test_date + pd.Timedelta(days=1):
    test_date + pd.Timedelta(days=7),
    "net_flow"
]

manual_result = future_flows.sum()
computed_result = target_df.loc[
    test_date, "target_7d_net_flow"
]

print("=== MANUAL VERIFICATION ===")
print("Prediction date:", test_date.date())

print("\nFuture daily net flows:")
print(future_flows)

print("\nManual calculation:", manual_result)
print("Computed target:", computed_result)

assert len(future_flows) == 7
assert manual_result == computed_result

print("\nTarget calculation verified.")

In [ ]:
# Step 17.3 — Target distribution


valid_target = target_df["target_7d_net_flow"].dropna()

print("=== SEVEN-DAY TARGET SUMMARY ===")
print(valid_target.describe())

print("\n=== TARGET CHARACTERISTICS ===")
print("Positive targets:", (valid_target > 0).sum())
print("Negative targets:", (valid_target < 0).sum())
print("Zero targets:", (valid_target == 0).sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(
    valid_target,
    bins=50,
    edgecolor="black",
    alpha=0.75
)
axes[0].axvline(0, linestyle="--")
axes[0].set_title("Distribution of Future 7-Day Net Flow")
axes[0].set_xlabel("Cumulative Net Flow (Events / 7 Days)")
axes[0].set_ylabel("Number of Prediction Dates")

axes[1].plot(
    target_df.index,
    target_df["target_7d_net_flow"]
)
axes[1].axhline(0, linestyle="--")
axes[1].set_title("Future 7-Day Net Flow Over Time")
axes[1].set_xlabel("Prediction Date")
axes[1].set_ylabel("Cumulative Net Flow (Events / 7 Days)")

plt.tight_layout()
plt.show()

In [ ]:
# Step 17.4 — Compare historical periods

periods = {
    "Pre-2020 (2014–2019)": ("2014-01-01", "2019-12-31"),
    "2020": ("2020-01-01", "2020-12-31"),
    "Post-2020 (2021–2024)": ("2021-01-01", "2024-12-31")
}

results = []

for name, (start, end) in periods.items():
    values = target_df.loc[
        start:end, "target_7d_net_flow"
    ].dropna()

    results.append({
        "Period": name,
        "Observations": len(values),
        "Mean": values.mean(),
        "Median": values.median(),
        "Std": values.std(),
        "Positive Rate": (values > 0).mean()
    })

period_summary = pd.DataFrame(results)

print("=== TARGET BY HISTORICAL PERIOD ===")
display(period_summary.round(3))

## 10. Baselines and preliminary time-aware validation
Baselines: (1) predict zero; (2) use the last seven observed daily net flows. Both use only information available at the end of decision day *t*. Validation uses 2024, with development labels ending on 2023-12-24 to avoid spanning the validation start. Because these rules are fixed rather than trained, the development set only defines the intended evaluation boundary. Overlapping validation targets mean the 359 daily errors are *not* independent replicates. Metrics describe this specific historical holdout and do not measure overcrowding prevention.

In [ ]:
# Step 18.1 — Construct two baseline predictions

baseline_df = target_df.copy()

# Baseline 1: Always predict zero future net flow
baseline_df["pred_zero"] = 0.0

# Baseline 2: Predict future flow using recent 7-day flow
baseline_df["pred_recent7"] = (
    baseline_df["net_flow"]
    .rolling(window=7, min_periods=7)
    .sum()
)

# Keep only rows with valid target and baseline predictions
baseline_valid = baseline_df.dropna(
    subset=[
        "target_7d_net_flow",
        "pred_recent7"
    ]
).copy()

print("=== BASELINE CONSTRUCTION ===")

print("Valid observations:", len(baseline_valid))

display(
    baseline_valid[
        ["net_flow", "target_7d_net_flow",
         "pred_zero", "pred_recent7"]
    ].head(10)
)

In [ ]:
# Step 18.2 — Chronological Baseline Validation


# Explicit chronological development and validation periods
dev = baseline_valid.loc[
    "2014-01-01":"2023-12-24"
].copy()

val = baseline_valid.loc[
    "2024-01-01":"2024-12-24"
].copy()

print("=== VALIDATION PERIODS ===")
print("Development:", dev.index.min(), "to", dev.index.max())
print("Validation:", val.index.min(), "to", val.index.max())
print("Development observations:", len(dev))
print("Validation observations:", len(val))

# Calculate baseline performance only on validation dates
y_true = val["target_7d_net_flow"]

results = []

for name, col in [
    ("Zero Net Flow", "pred_zero"),
    ("Recent 7-Day Flow", "pred_recent7")
]:
    y_pred = val[col]

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    results.append({
        "Baseline": name,
        "MAE": mae,
        "RMSE": rmse
    })

baseline_results = pd.DataFrame(results)

print("\n=== 2024 BASELINE PERFORMANCE ===")
display(baseline_results.round(3))

In [ ]:
# Optional focused diagnostic plot; evaluation metrics above use the full validation period.
plot_data = val.loc['2024-04-01':'2024-06-30']
plt.figure(figsize=(12, 4))
plt.plot(plot_data.index, plot_data['target_7d_net_flow'], label='Observed future 7-day net flow')
plt.plot(plot_data.index, plot_data['pred_recent7'], label='Recent-7 baseline')
plt.axhline(0, linestyle='--', label='Zero baseline')
plt.title('Baseline diagnostic: April–June 2024')
plt.xlabel('Prediction date')
plt.ylabel('Net events over seven days')
plt.legend()
plt.grid(alpha=.3)
plt.tight_layout()
plt.show()

## 11. Limitations, leakage, and next steps

- **Scope and measurement:** Animal-event net flow can support an initial analysis; it cannot independently establish animals physically housed onsite or historical kennel occupancy. A reliable historical capacity/onsite series is needed for a true overcrowding target.
- **Missingness and selection:** Missing outcomes, source-system migration, animal-type distinctions, and dates with zero recorded events require documented handling. Repeated Animal IDs and candidate unmatched events should not be interpreted as independent stays.
- **Dependence and distribution shift:** Seasonality, serial correlation, overlapping seven-day targets, and changed operations around 2020 make random row splits inappropriate.
- **Leakage controls:** In future modeling, fit normalization, target transformations, feature selection and hyperparameter tuning only within the appropriate training windows. Respect the end-of-day feature cutoff, and purge any development targets that enter a future validation window.
- **Future analytical work:** Validate inventory against additional independent official reports, clarify the onsite/offsite/capacity definitions, test robust baselines in rolling-origin validation, and only then consider a true overcrowding risk model.
- **Action and causality:** Predictions inform shelter managers; no intervention-effect or causal conclusion is established here.

### AI use disclosure
ChatGPT was used for code organization, analytical review, explanatory Markdown, and language editing. The numerical results were computed using the Austin Animal Center data. The notebook was run from a restarted kernel in the local development environment, as reported by the project author.